<a href="https://colab.research.google.com/github/Jinn-M4/Unified-Autonomous-Driving-System-2/blob/main/Unified_Autonomous_Driving_System_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
import os

folders = [
    "simulation",
    "perception",
    "system",
    "planning",
    "control"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/__init__.py", "w").close()  # 패키지 인식용

## Simulation

In [66]:
%%writefile simulation/vehicle.py
class Vehicle:
    def __init__(self, initial_speed=20.0, lead_distance=30.0):
        self.speed = initial_speed
        self.lead_distance = lead_distance
        self.lane_offset = 0.2
        self.lead_speed = 15.0
        self.drag_coeff = 0.05  # Add wind drag

    def update(self, accel, steering, dt=0.1):
        # Speed ​​change based on acceleration + resistance reflection
        self.speed += (accel - self.speed * self.drag_coeff) * dt
        self.speed = max(0, self.speed)

        # Distance update using relative speed
        relative_speed = self.speed - self.lead_speed
        self.lead_distance -= relative_speed * dt
        self.lane_offset += steering * dt

        if self.lead_distance < 0:
            self.lead_distance = 0
            self.speed = 0

Overwriting simulation/vehicle.py


## Perception

In [67]:
%%writefile perception/sensors.py
import random

def get_lane_offset(vehicle):
    noise = random.uniform(-0.05, 0.05)
    return vehicle.lane_offset + noise

def get_lead_distance(vehicle):
    cam_noise = random.uniform(-2.0, 2.0)
    radar_noise = random.uniform(-1.0, 1.0)

    cam_dist = vehicle.lead_distance + cam_noise
    radar_dist = vehicle.lead_distance + radar_noise

    return cam_dist, radar_dist

Overwriting perception/sensors.py


## System

In [68]:
%%writefile system/fusion.py
def fuse_distance(cam_dist, radar_dist):
    return 0.7 * radar_dist + 0.3 * cam_dist

Overwriting system/fusion.py


## Planning

In [69]:
%%writefile planning/behavior.py
class BehaviorPlanner:
    def __init__(self):
        self.state = "CRUISE"
        self.state_duration = 0
        self.MIN_STATE_DURATION = 5 # 상태 유지 시간 증가 (안정성)

    def compute_ttc(self, distance, ego_speed, lead_speed):
        rel_speed = ego_speed - lead_speed
        if rel_speed <= 0: return float('inf') # No risk of collision if the car in front is faster
        return distance / rel_speed

    def decide(self, distance, ego_speed, lead_speed):
        ttc = self.compute_ttc(distance, ego_speed, lead_speed)

        # logic decision
        if ttc < 1.8: # Ensure safety margin
            new_state = "EMERGENCY_BRAKE"
        elif ttc < 3.5:
            new_state = "SLOW_DOWN"
        elif distance < 35:
            new_state = "FOLLOW"
        else:
            new_state = "CRUISE"

        # State Smoothing (Chattering prevention)
        if new_state != self.state:
            if self.state_duration >= self.MIN_STATE_DURATION:
                self.state = new_state
                self.state_duration = 0

        self.state_duration += 1
        return self.state

Overwriting planning/behavior.py


## Control

In [70]:
%%writefile control/acc.py
import numpy as np

ACC_MAX = 2.5
BRAKE_NORMAL = -4.0
BRAKE_EMERGENCY = -9.0
TIME_HEADWAY = 2.2

class PIDController:
    def __init__(self, kp=1.5, ki=0.05, kd=0.3):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.integral = 0.0
        self.prev_error = 0.0
        self.prev_derivative = 0.0

    def control(self, error, dt=0.1):
        self.integral += error * dt
        self.integral = np.clip(self.integral, -15, 15)

        raw_derivative = (error - self.prev_error) / dt
        derivative = 0.7 * raw_derivative + 0.3 * self.prev_derivative

        output = self.kp * error + self.ki * self.integral + self.kd * derivative

        self.prev_error = error
        self.prev_derivative = derivative
        return output

class LongitudinalController:
    def __init__(self):
        self.pid = PIDController()
        self.TIME_HEADWAY = 2.0

    def get_accel(self, behavior, distance, speed, dt=0.1):
        if behavior == "EMERGENCY_BRAKE": return BRAKE_EMERGENCY
        if behavior == "SLOW_DOWN": return BRAKE_NORMAL
        if behavior == "CRUISE": return ACC_MAX

        # FOLLOW mode
        desired_dist = speed * self.TIME_HEADWAY + 5.0
        error = distance - desired_dist
        return np.clip(self.pid.control(error, dt), BRAKE_EMERGENCY, ACC_MAX)

# Add Rule-based function
def acc_control_rule(distance, speed, behavior):
    desired_distance = speed * TIME_HEADWAY + 5.0

    if behavior == "EMERGENCY_BRAKE":
        acc = BRAKE_EMERGENCY
    elif behavior == "SLOW_DOWN":
        acc = BRAKE_NORMAL
    elif behavior == "FOLLOW":
        # 단순 비례 제어 (P-제어와 유사)
        acc = 0.3 * (distance - desired_distance)
    else:
        acc = ACC_MAX

    return np.clip(acc, BRAKE_EMERGENCY, ACC_MAX)

Overwriting control/acc.py


In [71]:
%%writefile control/lka.py
def lka_control(offset):
    k = 0.5
    return -k * offset

Overwriting control/lka.py


## Main

In [72]:
%%writefile main.py
import time
import csv
import matplotlib.pyplot as plt
import numpy as np

from simulation.vehicle import Vehicle
from perception.sensors import get_lane_offset, get_lead_distance
from system.fusion import fuse_distance
from planning.behavior import BehaviorPlanner
from control.acc import LongitudinalController, acc_control_rule
from control.lka import lka_control

def run_simulation(use_pid=True):
    # 객체 인스턴스 생성 (상태 캡슐화)
    vehicle = Vehicle(initial_speed=20.0, lead_distance=30.0)
    planner = BehaviorPlanner()
    controller = LongitudinalController()

    dt = 0.1
    sim_steps = 100

    time_log, speed_log, distance_log, behavior_log, ttc_log = [], [], [], [], []

    for t in range(sim_steps):
        # scenario: The car in front braked suddenly 4 seconds later (v=0)
        if t == 40:
            vehicle.lead_speed = 0

        # 1. Perception & Fusion
        lane_offset = get_lane_offset(vehicle)
        cam_dist, radar_dist = get_lead_distance(vehicle)
        fused_distance = fuse_distance(cam_dist, radar_dist)

        # 2. Planning
        behavior = planner.decide(fused_distance, vehicle.speed, vehicle.lead_speed)

        # 3. Control
        if use_pid:
            # 클래스 내부 PID 사용
            accel = controller.get_accel(behavior, fused_distance, vehicle.speed, dt)
        else:
            # 기존 Rule-based 함수 호출
            accel = acc_control_rule(fused_distance, vehicle.speed, behavior)

        steering = lka_control(lane_offset)

        # 4. Vehicle Update (물리 반영)
        vehicle.update(accel, steering, dt)

        # save log data
        time_log.append(t * dt)
        speed_log.append(vehicle.speed)
        distance_log.append(vehicle.lead_distance)
        behavior_log.append(behavior)
        ttc = planner.compute_ttc(fused_distance, vehicle.speed, vehicle.lead_speed)
        ttc_log.append(ttc if ttc != float('inf') else 10.0) # 그래프 가독성을 위해 inf 제한

    return time_log, speed_log, distance_log, behavior_log, ttc_log

def analyze_performance(name, time_log, speed_log, distance_log, ttc_log):
    speed = np.array(speed_log)
    distance = np.array(distance_log)

    # 목표 거리 (Time Headway 2.0s + 여유 5m)
    desired_distance = speed * 2.0 + 5.0
    error = np.abs(distance - desired_distance)

    mean_error = np.mean(error)
    min_ttc = np.min(ttc_log)
    smoothness = np.mean(np.abs(np.diff(speed)))

    print(f"\n📊 [{name} Performance Review]")
    print(f"평균 거리 오차: {mean_error:.2f}m")
    print(f"최소 충돌 시간(TTC): {min_ttc:.2f}s")
    print(f"주행 부드러움(Accel 편차): {smoothness:.2f}")

    return {"error": mean_error, "ttc": min_ttc, "smooth": smoothness}

def compare_and_plot():
    # 데이터 획득
    pid_res = run_simulation(use_pid=True)
    rule_res = run_simulation(use_pid=False)

    # 지표 분석
    p_metrics = analyze_performance("PID Control", *pid_res[0:3], pid_res[4])
    r_metrics = analyze_performance("Rule-based", *rule_res[0:3], rule_res[4])

    # 시각화
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # 속도 및 거리 비교
    ax1.plot(pid_res[0], pid_res[1], 'b-', label="PID Speed")
    ax1.plot(rule_res[0], rule_res[1], 'r--', label="Rule Speed")
    ax1.set_title("Speed Tracking Comparison")
    ax1.set_ylabel("Speed (m/s)")
    ax1.legend()
    ax1.grid(True)

    # TTC 비교 (안전성)
    ax2.plot(pid_res[0], pid_res[4], 'b-', label="PID TTC")
    ax2.plot(rule_res[0], rule_res[4], 'r--', label="Rule TTC")
    ax2.axhline(y=1.5, color='orange', linestyle=':', label="Danger Zone")
    ax2.set_title("Safety Metric: TTC")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("TTC (s)")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig("refined_comparison.png")
    plt.show()

if __name__ == "__main__":
    compare_and_plot()

Overwriting main.py


In [73]:
!python main.py


📊 [PID Control Performance Review]
평균 거리 오차: 2.92m
최소 충돌 시간(TTC): 2.58s
주행 부드러움(Accel 편차): 0.31

📊 [Rule-based Performance Review]
평균 거리 오차: 4.48m
최소 충돌 시간(TTC): 2.95s
주행 부드러움(Accel 편차): 0.22
Figure(1200x1000)
